# Глава 13: Дообучение модели

> «Удача приходит, когда подготовка встречается с возможностью.»
> — **Сенека**

---

## Что вы узнаете

- Когда выбирать дообучение вместо улучшенных промптов
- Как подготовить обучающие данные в формате инструкция/ответ
- Почему маскирование потерь фокусирует обучение на важном
- Как адаптеры LoRA достигают эффективности 48× по параметрам
- Оценка поведения модели до и после настройки

---

## Установка

Сначала установим необходимые пакеты и проверим доступность GPU.

In [ ]:
# Установка проверенных версий
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== ИМПОРТЫ =====
import math
import json
import os
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer
from tqdm import tqdm

# Проверка GPU
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Используется устройство: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} ГБ")
else:
    print("ВНИМАНИЕ: GPU не обнаружен. Обучение будет медленным.")
    print("Перейдите в Runtime > Change runtime type > GPU")

In [ ]:
# ===== ВОСПРОИЗВОДИМОСТЬ =====
import random
import numpy as np

def set_seed(seed=42):
    """Установка семян всех источников случайности (тот же помощник, что в главе 12)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)


## 1. Компоненты модели из глав 10-12

Сначала импортируем модель MiniGPT, которую мы построили в предыдущих главах.

In [ ]:
# ===== МНОГОГОЛОВОЕ ВНИМАНИЕ (из главы 10) =====
class MultiHeadAttention(nn.Module):
    """
    Эффективное многоголовое внимание (все головы обрабатываются вместе).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model должен делиться на num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Объединённая QKV проекция (в 3 раза эффективнее отдельных!)
        # Почему 3 * d_model? Потому что мы проецируем в Q, K, V одновременно
        # bias=True соответствует GPT-2, поэтому позже мы сможем загрузить его предобученные смещения
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Выходная проекция
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout на весах внимания
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Шаг 1: Проекция в Q, K, V (всё сразу!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Шаг 2: Разделение на Q, K, V и изменение формы для многоголового =====
        # Изменяем форму на (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Переставляем в (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Разделяем на Q, K, V: каждый имеет форму (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Шаг 3: Масштабированное скалярное произведение внимания (батчи по головам) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Применяем причинную маску, если она предоставлена
        if mask is not None:
            # Расширяем маску для голов: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Взвешенная сумма значений
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Шаг 4: Конкатенация голов =====
        # Транспонируем в (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Изменяем форму на (batch, seq, d_model), это конкатенирует головы
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Шаг 5: Финальная проекция =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Тестируем

print("MultiHeadAttention определён!")

In [ ]:
# ===== ПОЛНОСВЯЗНАЯ СЕТЬ (из главы 10) =====
class FeedForward(nn.Module):
    """
    Позиционная полносвязная сеть.
    Применяется к каждой позиции независимо (одинаковые веса для всех позиций).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768 для GPT-2 small)
            d_ff: Размерность полносвязной сети (обычно 4 * d_model = 3072)
            dropout: Вероятность dropout
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), расширение
        x = F.gelu(x, approximate="tanh")  # Нелинейность (вариант GELU из GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), проекция обратно
        return x


# Тестируем

print("FeedForward определён!")

In [ ]:
# ===== БЛОК ТРАНСФОРМЕРА (из главы 10) =====
class TransformerBlock(nn.Module):
    """
    Полный блок трансформера с многоголовым вниманием, полносвязной сетью,
    остаточными соединениями и нормализацией слоя (стиль пре-нормализации, как в GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768)
            num_heads: Количество голов внимания (12)
            d_ff: Размерность полносвязной сети (3072 = 4 * d_model)
            dropout: Вероятность dropout
        """
        super().__init__()

        # Нормализация слоя (перед каждым подслоем)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Многоголовое внимание
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Полносвязная сеть
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (применяется после каждого подслоя)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: Входные эмбеддинги (batch, seq, d_model)
            mask: Причинная маска (seq, seq) или (1, 1, seq, seq)

        Returns:
            x: Выходные эмбеддинги (batch, seq, d_model)
            attn_weights: Веса внимания (batch, heads, seq, seq)
        """
        # ===== Многоголовое внимание с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД вниманием
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Остаточное соединение

        # ===== Полносвязная сеть с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД полносвязной сетью
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Остаточное соединение

        return x, attn_weights


# Тестируем полный блок трансформера

print("TransformerBlock определён!")

In [ ]:
# ===== КОНФИГУРАЦИЯ GPT (из главы 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Конфигурация для модели MiniGPT."""
    vocab_size: int = 50257      # Размер словаря GPT-2
    max_seq_len: int = 1024      # Максимальная длина контекста
    embed_dim: int = 768         # Размерность эмбеддинга (d_model)
    num_heads: int = 12          # Количество голов внимания
    num_layers: int = 12         # Количество блоков трансформера
    d_ff: int = 3072             # Размерность скрытого слоя полносвязной сети (4 * embed_dim)
    dropout: float = 0.1         # Вероятность дропаута

    def __post_init__(self):
        """Проверка конфигурации после инициализации."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) должен делиться на " \
            f"num_heads ({self.num_heads})"

In [ ]:
# ===== МОДЕЛЬ MINIGPT (из главы 11) =====
class MiniGPT(nn.Module):
    """
    Минимальная языковая модель в стиле GPT.
    Объединяет эмбеддинги, блоки трансформера и голову языкового моделирования.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Эмбеддинги токенов и позиций (глава 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Блоки трансформера (глава 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Финальная нормализация слоя =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Голова языкового моделирования =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Связывание весов =====
        # Общие веса между эмбеддингом и выходной проекцией
        self.lm_head.weight = self.token_embed.weight

        # Инициализация весов
        self._init_weights()

    def _init_weights(self):
        """Инициализация весов небольшими случайными значениями."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Прямой проход через модель.

        Args:
            token_ids: ID входных токенов (batch, seq)
            return_attention: Возвращать ли веса внимания

        Returns:
            logits: Оценки для словаря (batch, seq, vocab_size)
            attention_weights: Необязательный список весов внимания по блокам
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Эмбеддинги =====
        # Эмбеддинги токенов
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Позиционные эмбеддинги
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Объединяем и применяем dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Создание причинной маски =====
        # Нижнетреугольная: каждая позиция может смотреть только на предыдущие позиции
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Проход через блоки трансформера =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Финальная нормализация слоя =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Голова языкового моделирования =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Генерация токенов, продолжающих input_ids (промпт включён в вывод).

        Args:
            input_ids: ID токенов промпта (batch, seq)
            max_new_tokens: Сколько новых токенов сгенерировать
            temperature: Температура сэмплирования (используется только при do_sample=True)
            do_sample: Сэмплировать из распределения; False = жадный argmax
            eos_token_id: Остановиться раньше, если каждая последовательность сгенерировала этот токен

        Returns:
            ID токенов (batch, seq + новые токены)
        """
        self.eval()

        # Позиционные эмбеддинги покрывают только max_seq_len позиций
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Промпт ({input_ids.shape[1]} токенов) + max_new_tokens "
                f"({max_new_tokens}) превышает контекст модели в {limit} "
                "токенов. Сократите промпт или уменьшите max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids

## 2. Получаем базовую модель главы 12 (импорт чекпоинта)

Книга дообучает **шекспировский чекпоинт главы 12**, и этот ноутбук тоже. Базовая оценка, обучение LoRA и перезагрузка адаптера используют **одни и те же базовые веса**. Два честных пути:

- **Путь A (предпочтительный):** загрузите свой `best_model.pt`, обученный в главе 12 (панель Files в Colab, или раскомментируйте две строки ниже).
- **Путь B (свежий Colab без файла):** ячейка ниже выполняет рецепт главы 12 как **мини-обучение на 1 эпоху** на TinyShakespeare и сохраняет `best_model.pt`. Эта база слабее 3-эпохового прогона книги; потери будут отличаться, но все ячейки используют одну согласованную базу.

Случайно инициализированная база — *отдельный, явно помеченный эксперимент* в конце, никогда не путь по умолчанию.

In [ ]:
# ===== ИМПОРТ ЧЕКПОИНТА ГЛАВЫ 12 =====
# Все ячейки ниже (база, обучение LoRA, перезагрузка) используют ЭТУ базу.
import os
import urllib.request

config = GPTConfig(
    vocab_size=50257, max_seq_len=128, embed_dim=256,
    num_heads=4, num_layers=4, d_ff=1024, dropout=0.1
)

# Быстрый токенизатор возвращает символьные смещения для маскирования потерь.
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Маршрут A: загрузите в Colab только свой чекпоинт из главы 12.
# from google.colab import files
# files.upload()  # выберите свой best_model.pt

if not os.path.exists("best_model.pt"):
    # Маршрут B: если чекпоинта нет, обучаем базу одну эпоху по рецепту главы 12.
    # Фрагменты по 512 символов, батчи по 32, AdamW с линейным разогревом.
    # Позиции заполнения исключаем из потерь по маске внимания.
    print("Нет best_model.pt: обучаем базу одну эпоху по рецепту главы 12…")
    from transformers import get_linear_schedule_with_warmup

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "shakespeare.txt")
    text = open("shakespeare.txt").read()

    def create_lm_labels(token_ids):
        inputs = token_ids[:, :-1]
        targets = token_ids[:, 1:].clone()
        return inputs, targets  # заполнение маскируется в цикле по attention_mask

    class TextDataset(Dataset):
        def __init__(self, text, chunk_size=512):
            self.chunks = [text[i:i + chunk_size]
                           for i in range(0, len(text) - chunk_size + 1, chunk_size)]
        def __len__(self):
            return len(self.chunks)
        def __getitem__(self, idx):
            return self.chunks[idx]

    def ch12_collate(batch, tokenizer, max_length=128):
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"]

    ds = TextDataset(text, chunk_size=512)
    n_train = int(0.9 * len(ds))
    train_ds, val_ds = torch.utils.data.random_split(
        ds, [n_train, len(ds) - n_train],
        generator=torch.Generator().manual_seed(42))
    collate12 = partial(ch12_collate, tokenizer=tokenizer, max_length=128)
    train_loader12 = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate12)

    set_seed(42)
    model = MiniGPT(config).to(device)
    optimizer = AdamW(model.parameters(), lr=3e-4)
    total_steps12 = len(train_loader12)
    warmup_steps12 = min(100, max(1, total_steps12 // 10))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps12,
        num_training_steps=total_steps12)

    model.train()
    total_loss = 0
    for input_ids, attention_mask in tqdm(train_loader12, desc="Обучение базы"):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # не учитываем заполнение в потерях
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               targets.reshape(-1), ignore_index=-100)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Средние потери за эпоху: {total_loss / len(train_loader12):.4f}")
    # Сохраняем только веса, без состояния оптимизатора и генератора случайных чисел.
    torch.save({"model_state_dict": model.state_dict()}, "best_model.pt")
    print("Сохранён best_model.pt: после одной эпохи база слабее, чем после трёх в книге.")

# В обоих маршрутах все дальнейшие сравнения используют один и тот же чекпоинт.
# Чекпоинт главы 12 содержит состояние генератора NumPy с типом uint32.
# Разрешаем только необходимые типы и только для этой загрузки.
import numpy as np
with torch.serialization.safe_globals([
    np._core.multiarray._reconstruct, np.ndarray, np.dtype, np.dtypes.UInt32DType,
]):
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=True)
base_model = MiniGPT(config).to(device)
base_model.load_state_dict(checkpoint["model_state_dict"])
base_model.eval()
BASE_SOURCE = "Чекпоинт главы 12 (best_model.pt)"
print(f"Базовая модель готова: {sum(p.numel() for p in base_model.parameters()):,} параметров")
print(f"Базовые веса: {BASE_SOURCE}")

## 3. Подготовка набора данных для дообучения

Создадим пары инструкция/ответ для FAQ компании TechStartup Inc.

Проверяем новые формулировки вопросов по знакомым темам: сами ответы уже встречались при обучении.

In [ ]:
# Русский FAQ: отделяем целые группы перефразировок.
import json
import random

# Семь написанных вручную пар FAQ для нашей вымышленной компании
base_examples = [
    {"instruction": "Чем занимается TechStartup Inc?",
     "response": "TechStartup Inc создаёт инструменты на базе ИИ."},

    {"instruction": "Когда была основана TechStartup Inc?",
     "response": "TechStartup Inc была основана в 2020 году."},

    {"instruction": "Где находится TechStartup Inc?",
     "response": "TechStartup Inc находится в Амстердаме."},

    {"instruction": "Что такое SmartScheduler?",
     "response": "SmartScheduler — это наш календарный помощник на базе ИИ."},

    {"instruction": "Сколько стоит SmartScheduler?",
     "response": "10 евро/месяц индивидуально, 8 евро/пользователь для команд."},

    {"instruction": "Как сбросить пароль?",
     "response": "Нажмите 'Забыли пароль' на странице входа."},

    {"instruction": "Могу ли я экспортировать свои данные?",
     "response": "Да, перейдите в Настройки > Данные > Экспорт."},
]

# Перефразировки для каждого вопроса (в том же порядке, что base_examples)
paraphrases = [
    ["Чем занимается TechStartup Inc?", "Каков бизнес TechStartup Inc?",
     "Расскажите, что производит TechStartup Inc.", "Что за компания TechStartup Inc?"],
    ["Когда была основана TechStartup Inc?", "В каком году появилась TechStartup Inc?",
     "Сколько лет TechStartup Inc?", "Когда TechStartup Inc начала работу?"],
    ["Где находится TechStartup Inc?", "Где базируется TechStartup Inc?",
     "В каком городе находится TechStartup Inc?", "Где расположены офисы TechStartup Inc?"],
    ["Что такое SmartScheduler?", "Расскажите о SmartScheduler.",
     "Что делает SmartScheduler?", "Опишите SmartScheduler."],
    ["Сколько стоит SmartScheduler?", "Какова цена SmartScheduler?",
     "Сколько SmartScheduler стоит в месяц?", "Сколько стоит подписка на SmartScheduler?"],
    ["Как сбросить пароль?", "Я забыл пароль, что делать?",
     "Как можно сменить пароль?", "Каков процесс сброса пароля?"],
    ["Могу ли я экспортировать свои данные?", "Как экспортировать мои данные?",
     "Возможен ли экспорт данных?", "Где находится опция экспорта данных?"],
]

def expand_faq(base_examples, paraphrases, seed=42):
    """Детерминированно расширить пары FAQ, разбивая по ГРУППАМ перефразировок.

    Каждая перефразировка вместе со своими четырьмя вариантами вежливого
    префикса образует одну группу (7 вопросов x 4 перефразировки = 28 групп
    по 4). Мы перемешиваем ГРУППЫ и откладываем 5 целых групп (20 примеров)
    ДО того, как какая-либо строка обучения/оценки появится в другом
    разделении, так что ни одна оценочная формулировка, с префиксом или без,
    никогда не попадала в обучение. Обучение берёт 80 из оставшихся 92.
    """
    prefixes = ["", "Пожалуйста, расскажите: ", "Быстрый вопрос: ", "Не могли бы вы рассказать: "]
    groups = []
    for qid, (example, variants) in enumerate(zip(base_examples, paraphrases)):
        for paraphrase in variants:
            groups.append((qid, paraphrase,
                           [{"instruction": p + paraphrase,
                             "response": example["response"]} for p in prefixes]))
    rng = random.Random(seed)
    rng.shuffle(groups)

    # Отложить 5 целых групп, но никогда ПОСЛЕДНЮЮ группу вопроса:
    # каждое намерение должно сохранить хотя бы одну формулировку в обучении.
    left = {qid: sum(1 for q, _, _ in groups if q == qid)
            for qid in range(len(base_examples))}
    eval_groups, train_groups = [], []
    for qid, paraphrase, examples in groups:
        if len(eval_groups) < 5 and left[qid] > 1:
            eval_groups.append((qid, paraphrase, examples))
        else:
            train_groups.append((qid, paraphrase, examples))
        left[qid] -= 1

    eval_data = [ex for _, _, g in eval_groups for ex in g]      # 5 x 4 = 20
    train_pool = [ex for _, _, g in train_groups for ex in g]    # 23 x 4 = 92
    rng.shuffle(train_pool)
    train_data = train_pool[:80]
    return train_data, eval_data

train_data, eval_data = expand_faq(base_examples, paraphrases)

# Проверка 1: точные строки вопросов не пересекаются (классическая проверка утечки)
train_questions = {ex["instruction"] for ex in train_data}
assert all(ex["instruction"] not in train_questions for ex in eval_data)

# Проверка 2: ГРУППЫ перефразировок не пересекаются. Снимаем вежливые префиксы
# и проверяем, что ни одна оценочная формулировка не встречается в обучении
# ни в какой префиксной форме.
def core_question(instruction):
    for p in ["Пожалуйста, расскажите: ", "Быстрый вопрос: ", "Не могли бы вы рассказать: "]:
        if instruction.startswith(p):
            return instruction[len(p):]
    return instruction

train_cores = {core_question(q) for q in train_questions}
assert all(core_question(ex["instruction"]) not in train_cores for ex in eval_data)

# При seed=42 в обучении есть все семь намерений, а в оценке — пять новых
# формулировок для четырёх из них. Новых намерений здесь нет.
train_intents = {ex["response"] for ex in train_data}
eval_intents = {ex["response"] for ex in eval_data}
print(f"Обучение: {len(train_data)}, Оценка: {len(eval_data)}")
print(f"Намерений покрыто в обучении: {len(train_intents)}/7")
print(f"Намерений в оценке: {len(eval_intents)}/7; из них в обучении: "
      f"{len(eval_intents & train_intents)}")
print(f"Оценочных формулировок встречено в обучении (префиксы сняты): "
      f"{sum(core_question(ex['instruction']) in train_cores for ex in eval_data)}/20")
# Вывод: Обучение: 80, Оценка: 20
# Вывод: Намерений покрыто в обучении: 7/7
# Вывод: Намерений в оценке: 4/7; из них в обучении: 4
# Вывод: Оценочных формулировок встречено в обучении (префиксы сняты): 0/20

def save_as_jsonl(examples, filepath):
    """Сохранить примеры в формате JSONL (один JSON-объект на строку)."""
    with open(filepath, 'w') as f:
        for example in examples:
            f.write(json.dumps(example) + '\n')

def load_jsonl(filepath):
    """Загрузить примеры из формата JSONL."""
    examples = []
    with open(filepath, 'r') as f:
        for line in f:
            examples.append(json.loads(line))
    return examples

# Сохраняем наш датасет (английский пример в архиве: data/faq_dataset.jsonl)
save_as_jsonl(train_data + eval_data, 'faq_dataset.jsonl')


In [ ]:
# ===== ФОРМАТИРОВАНИЕ + БАТЧИ (из книги) =====
# Определяем специальные токены
INST_START = "[INST]"
INST_END = "[/INST]"

def build_prompt(instruction):
    """ЕДИНЫЙ формат промпта, общий для обучения и генерации.

    Заканчивается сразу после [/INST], без завершающего пробела:
    завершающий пробел может токенизироваться отдельно иначе, чем
    склеенным с первым словом ответа (" Tech"), и тогда инференс
    подавал бы модели префикс, которого она не видела в обучении.
    """
    return f"{INST_START} {instruction} {INST_END}"

def format_example(instruction, response, tokenizer):
    """
    Форматировать пару инструкция/ответ для обучения.

    Нужен быстрый токенизатор (GPT2TokenizerFast, который загружает наша настройка):
    только быстрые токенизаторы сообщают символьные смещения для каждого токена.

    Возвращает:
        input_ids: Полная токенизированная последовательность
        labels: То же, что input_ids, но с -100 для токенов инструкции
    """
    # Формат: [INST] инструкция [/INST] ответ + конец текста.
    # Токен EOS учит модель, где ответы ЗАКАНЧИВАЮТСЯ, чтобы генерация
    # могла остановиться там, а не тараторить дальше после ответа.
    prompt = build_prompt(instruction)
    full_text = prompt + " " + response + tokenizer.eos_token

    # Токенизируем, запрашивая символьный диапазон каждого токена в full_text
    enc = tokenizer(full_text, return_offsets_mapping=True)
    input_ids = torch.tensor(enc["input_ids"])

    # Находим, где начинается ответ: первый токен, чей символьный
    # диапазон целиком лежит за промптом. BPE приклеивает добавленный
    # нами пробел к первому слову ответа (" Tech"), поэтому этот токен
    # принадлежит ответу; подсчёт одних токенов промпта замаскировал бы его.
    response_start = next(i for i, (_, end) in enumerate(enc["offset_mapping"])
                          if end > len(prompt))

    # Создаём метки: -100 для токенов инструкции (игнорируются функцией потерь)
    labels = input_ids.clone()
    labels[:response_start] = -100  # Маскируем токены инструкции

    return input_ids, labels


import torch
from functools import partial
from torch.utils.data import DataLoader

PAD_ID = tokenizer.eos_token_id  # Для заполнения коротких последовательностей используем ID токена конца текста.

def collate_fn(examples, tokenizer, max_length=128):
    """Токенизирует пары «инструкция — ответ» и заполняет input_ids и labels до общей длины."""
    batch_ids, batch_labels = [], []
    for ex in examples:
        ids, labels = format_example(ex['instruction'], ex['response'], tokenizer)
        ids, labels = ids[:max_length], labels[:max_length]
        pad = max_length - len(ids)
        ids = torch.cat([ids, torch.full((pad,), PAD_ID, dtype=torch.long)])
        labels = torch.cat([labels, torch.full((pad,), -100, dtype=torch.long)])
        batch_ids.append(ids)
        batch_labels.append(labels)
    return {'input_ids': torch.stack(batch_ids),
            'labels': torch.stack(batch_labels)}

collate = partial(collate_fn, tokenizer=tokenizer)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, collate_fn=collate)


# Проверка одного батча
batch = next(iter(train_loader))
print(f"Форма input IDs: {batch['input_ids'].shape}")
print(f"Форма меток: {batch['labels'].shape}")
print(f"Маскированных позиций в первом примере: {(batch['labels'][0] == -100).sum().item()}")

## 4. Базовая оценка (ДО дообучения)

Посмотрим, как базовая модель отвечает на наши промпты FAQ.

In [ ]:
@torch.no_grad()
def generate_response(model, prompt, tokenizer, max_new_tokens=50, temperature=0.8):
    """Генерация ответа на промпт с инструкцией."""
    model.eval()
    full_prompt = build_prompt(prompt)
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)
    
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    
    response = tokenizer.decode(output_ids[0][len(input_ids[0]):])
    return response.strip()

# Тестирование на промптах FAQ
test_prompts = [
    "Чем занимается TechStartup Inc?",
    "Как сбросить пароль?",
    "Что такое SmartScheduler?"
]

print(f"ДО ДООБУЧЕНИЯ ({BASE_SOURCE}):")
print("="*60)
for prompt in test_prompts:
    response = generate_response(base_model, prompt, tokenizer)
    print(f"\nВопрос: {prompt}")
    print(f"Ответ: {response[:100]}...")
print("\n(Бессмыслица в духе Шекспира - база главы 12 ничего не знает")
print("о TechStartup Inc, а модель на 16M с нуля никогда не учила")
print("беглый английский. Честные ожидания!)")


## 5. Определение компонентов LoRA

Создадим слой адаптера LoRA, который добавляет обучаемые параметры без изменения базовых весов.

### Почему LoRA вместо полного дообучения?

У полного дообучения есть проблема: **катастрофическое забывание**. Когда вы обновляете все веса для новой задачи, модель может "забыть" то, что выучила во время предобучения. LoRA значительно уменьшает это, сохраняя базовые веса *замороженными* и добавляя небольшие обучаемые матрицы *рядом* с ними.

Думайте об этом как об очках для чтения: ваши глаза (базовая модель) остаются прежними, но очки (LoRA) добавляют небольшую коррекцию для конкретных задач.

In [ ]:
# ===== ЛИНЕЙНЫЙ СЛОЙ LORA (конвенция статьи: A вниз, B вверх) =====
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    Линейный слой с низкоранговой адаптацией (LoRA).

    Думайте об этом как о добавлении небольшой «корректирующей заметки» к существующему слою.
    Базовый слой остаётся замороженным, а две маленькие матрицы учат поправки.
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()

        # Исходный слой (мы его заморозим); смещение соответствует qkv_proj GPT-2
        self.base = nn.Linear(in_features, out_features)

        # Матрицы LoRA (конвенция статьи: A вниз, B вверх)
        # A: (in_features, rank) — проекция «вниз»
        # B: (rank, out_features) — проекция «вверх»
        self.lora_A = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

        # Инициализация A небольшими случайными значениями (Kaiming), B — нулями.
        # Инициализируем через транспонированное представление, чтобы Kaiming
        # видел (rank, in_features) — форму веса nn.Linear(in_features, rank):
        # fan_in тогда равен in_features. Запись на месте в представление
        # обновляет сам lora_A.
        nn.init.kaiming_uniform_(self.lora_A.T, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        # Фактор масштабирования
        self.scale = alpha / rank

    def freeze_base(self):
        """Заморозить базовый слой, чтобы обучались только матрицы LoRA."""
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

    def forward(self, x):
        # форма x: (batch, sequence, in_features)

        # Исходный выход из замороженных весов
        base_output = self.base(x)  # (batch, seq, out_features)

        # Путь LoRA: x @ A @ B, с масштабированием
        # x @ A: (batch, seq, in_features) @ (in_features, rank)
        #      = (batch, seq, rank)  [сжато!]
        # ... @ B: (batch, seq, rank) @ (rank, out_features)
        #        = (batch, seq, out_features)  [расширено обратно]
        lora_output = (x @ self.lora_A @ self.lora_B) * self.scale

        return base_output + lora_output

print("LoRALinear определён!")

# Демонстрация экономии параметров
in_features, out_features, rank = 256, 256, 8
full_params = in_features * out_features
lora_params = in_features * rank + rank * out_features

print(f"\nСравнение параметров (слой 256x256):")
print(f"  Полная тонкая настройка: {full_params:,} параметров")
print(f"  LoRA (rank=8):           {lora_params:,} параметров")
print(f"  Экономия:                {full_params/lora_params:.1f}x меньше!")

## 6. Применение LoRA к модели

Заменим проекции внимания на версии LoRA и заморозим базовую модель.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16):
    """
    Добавление LoRA к проекциям внимания.

    Мы нацеливаемся на qkv_proj — объединённую проекцию query/key/value.
    Адаптер учит совместные коррекции для Q, K и V.
    """
    # Шаг 1: Сначала заморозить ВСЕ параметры
    for param in model.parameters():
        param.requires_grad = False

    # Шаг 2: Заменить qkv_proj в каждом блоке трансформера
    # Мы явно перебираем блоки, которые построили в главе 11
    for block in model.blocks:
        # Получаем проекцию QKV слоя внимания
        original_qkv = block.attn.qkv_proj
        in_features = original_qkv.in_features
        out_features = original_qkv.out_features

        # Создаём замену с LoRA на том же устройстве, что и модель
        lora_qkv = LoRALinear(in_features, out_features, rank, alpha)
        lora_qkv = lora_qkv.to(original_qkv.weight.device)

        # Копируем предобученные веса (и смещение!) в базовый слой
        lora_qkv.base.weight.data = original_qkv.weight.data.clone()
        lora_qkv.base.bias.data = original_qkv.bias.data.clone()

        # Замораживаем базу (матрицы LoRA остаются обучаемыми)
        lora_qkv.freeze_base()
        if lora_qkv.base.bias is not None:
            lora_qkv.base.bias.requires_grad = False

        # Заменяем слой
        block.attn.qkv_proj = lora_qkv

    return model


# Та же база, что в книге: импортированный выше чекпоинт главы 12.
# База, обучение и перезагрузка делят эти веса, поэтому сравнения
# ДО/ПОСЛЕ остаются честными. (При мини-обучении пути B потери
# будут отличаться от чисел 3-эпохового чекпоинта книги.)
set_seed(42)  # ДО инициализации адаптеров и перемешивания батчей (посев после обучения ничего не воспроизводит)
model = MiniGPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model = add_lora_to_model(model, rank=8, alpha=16)

# Подсчёт параметров
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Всего параметров:      {total:,}")
print(f"Обучаемых (LoRA):      {trainable:,}")
print(f"Процент обучаемых:     {100*trainable/total:.2f}%")

## 7. Цикл дообучения

Обучение с использованием 5-шагового рецепта с маскированными потерями.

In [ ]:
import torch.nn.functional as F

def compute_masked_loss(logits, labels):
    """
    Вычисление потерь кросс-энтропии, игнорируя позиции, где labels == -100.

    Это как оценивать только часть ответа на экзамене,
    а не вопрос, который был скопирован из промпта.
    """
    # Сдвиг для предсказания следующего токена (как в главе 12)
    # .contiguous() обеспечивает последовательное расположение памяти для .view()
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    # Кросс-энтропия с ignore_index=-100
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100  # PyTorch игнорирует эти позиции
    )

    return loss


def finetune_epoch(model, dataloader, optimizer, device):
    """
    Дообучение на одну эпоху по 5-шаговому рецепту с маскированными потерями.
    """
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        # ===== 5-ШАГОВЫЙ РЕЦЕПТ =====

        # Шаг 1: Обнуление градиентов
        optimizer.zero_grad()

        # Шаг 2: Прямой проход
        logits = model(input_ids)

        # Шаг 3: Вычисление МАСКИРОВАННЫХ потерь
        loss = compute_masked_loss(logits, labels)

        # Шаг 4: Обратный проход
        loss.backward()

        # Шаг 5: Обновление весов
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

print("Функции обучения определены!")

In [ ]:
# ===== ОБУЧЕНИЕ =====
num_epochs = 3
learning_rate = 1e-3  # Адаптеры LoRA допускают более высокую скорость обучения, чем полное дообучение

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate  # same optimizer as the book: plain AdamW, no weight decay
)

train_losses = []
print(f"Обучение на {len(train_data)} примерах в течение {num_epochs} эпох\n")
for epoch in range(num_epochs):
    loss = finetune_epoch(model, train_loader, optimizer, device)
    train_losses.append(loss)
    print(f"Эпоха {epoch+1}: маскированные потери {loss:.4f}")

print("\nДообучение завершено!")
print("Примечание: с базовой моделью на 16M, обученной с нуля, ожидайте бессмысленные генерации;")
print("выигрыш — это снижение потерь и механика адаптеров. Для связных ответов")
print("начните с предобученных весов GPT-2 (загрузчик из главы 11).")

## 8. Визуализация прогресса обучения

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Эпоха', fontsize=12)
plt.ylabel('Потери обучения', fontsize=12)
plt.title('Кривая потерь при дообучении', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.show()

print(f"\nПотери уменьшились с {train_losses[0]:.4f} до {train_losses[-1]:.4f}")
print(f"Улучшение: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 9. Оценка (ПОСЛЕ дообучения)

Честная часть: измеряем, что изменилось. С нашей базовой моделью на 16M, обученной с нуля, генерации остаются бессмыслицей и ДО, и ПОСЛЕ (она так и не выучила связный английский). Измеримый выигрыш — снижение потерь на обучении и механика адаптеров. Связное «до/после» даёт предобученная база вроде GPT-2 (загрузчик из главы 11).

In [ ]:
def evaluate_model(model, test_examples, tokenizer, device):
    """
    Оценить дообученную модель на тестовых примерах.
    """
    model.eval()
    results = []

    for example in test_examples:
        instruction = example['instruction']
        expected = example['response']

        # Генерируем из ТОГО ЖЕ формата префикса, что использовало обучение
        prompt = build_prompt(instruction)
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

        # Потери отложенной выборки: прогоняем ПРАВИЛЬНЫЙ ответ через модель
        # с учительским форсированием и оцениваем только токены ответа
        # (та же маскированная функция потерь, что в обучении, но на формулировках вопросов,
        # которых адаптеры никогда не видели)
        full_ids, full_labels = format_example(instruction, expected, tokenizer)
        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(device))
            resp_loss = compute_masked_loss(logits, full_labels.unsqueeze(0).to(device))

        output = model.generate(
            input_ids,
            max_new_tokens=50,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

        # Декодируем только новые токены, отбрасывая сам терминатор EOS
        new_tokens = output[0][input_ids.shape[1]:].tolist()
        if tokenizer.eos_token_id in new_tokens:
            new_tokens = new_tokens[:new_tokens.index(tokenizer.eos_token_id)]
        generated = tokenizer.decode(new_tokens).strip()

        # Точное совпадение
        is_exact = generated.lower() == expected.lower()

        # Пересечение слов
        gen_words = set(generated.lower().split())
        exp_words = set(expected.lower().split())
        overlap = len(gen_words & exp_words) / max(len(exp_words), 1)

        results.append({
            'instruction': instruction,
            'expected': expected,
            'generated': generated,
            'exact_match': is_exact,
            'word_overlap': overlap,
            'response_loss': resp_loss.item()
        })

    # Сводка
    accuracy = sum(r['exact_match'] for r in results) / len(results)
    avg_overlap = sum(r['word_overlap'] for r in results) / len(results)

    return {
        'exact_match_accuracy': accuracy,
        'average_word_overlap': avg_overlap,
        'avg_response_loss': sum(r['response_loss'] for r in results) / len(results),
        'detailed_results': results
    }

# Оценка на отложенной выборке
print("Оценка на тестовом наборе...")
results = evaluate_model(model, eval_data, tokenizer, device)

print(f"\n===== РЕЗУЛЬТАТЫ ОЦЕНКИ =====")
print(f"Точность точных совпадений: {results['exact_match_accuracy']*100:.1f}%")
print(f"Среднее пересечение слов: {results['average_word_overlap']*100:.1f}%")
print("(Около 0% ожидается с базовой моделью, обученной с нуля; см. примечание выше.)")
print(f"Потери на ответах holdout: {results['avg_response_loss']:.2f}")


In [ ]:
# Показываем детальные результаты
print("\n===== ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ ТЕСТА =====")
for r in results['detailed_results']:
    status = "✓" if r['exact_match'] else "○"
    print(f"\n{status} Вопрос: {r['instruction']}")
    print(f"  Ожидалось:   {r['expected']}")
    print(f"  Сгенерировано: {r['generated'][:100]}..." if len(r['generated']) > 100 else f"  Сгенерировано: {r['generated']}")
    print(f"  Пересечение: {r['word_overlap']*100:.0f}%")

In [ ]:
def show_before_after(base_model, finetuned_model, test_prompts, tokenizer, device):
    """
    Сравнивает ответы базовой и дообученной модели на одинаковые вопросы.
    """
    print("=" * 70)
    print("До и после дообучения")
    print("=" * 70)

    for prompt in test_prompts:
        full_prompt = build_prompt(prompt)
        input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)

        base_output = base_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)
        ft_output = finetuned_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)

        base_text = tokenizer.decode(base_output[0][len(input_ids[0]):])
        ft_text = tokenizer.decode(ft_output[0][len(input_ids[0]):])

        print(f"\nПромпт: {prompt}")
        print(f"  ДО: {base_text.strip()}")
        print(f"  ПОСЛЕ:  {ft_text.strip()}")
        print("-" * 70)

# Модель ДО: точная копия базовых весов (без адаптации)
base_model_fresh = MiniGPT(config).to(device)
base_model_fresh.load_state_dict(base_model.state_dict())

test_prompts = ["Чем занимается TechStartup Inc?", "Как сбросить пароль?"]
set_seed(42)  # повторный посев, чтобы сравнение с сэмплированием ниже было воспроизводимым
show_before_after(base_model_fresh, model, test_prompts, tokenizer, device)

print("\nОбе стороны — бессмыслица: модель на 16M, обученная с нуля, так и не выучила")
print("связный английский. Измеримый выигрыш — снижение потерь + рабочая механика адаптеров.")
print("Сначала загрузите веса GPT-2 (глава 11), чтобы увидеть связное «до/после».")

## 10. Сохранение и загрузка весов LoRA

In [ ]:
def save_lora_weights(model, filepath):
    """
    Сохранение только параметров LoRA (крошечный файл!).
    """
    lora_state_dict = {
        name: param for name, param in model.state_dict().items()
        if 'lora_' in name
    }
    torch.save(lora_state_dict, filepath)

    # Показываем размер файла
    import os
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"Веса LoRA сохранены: {filepath} ({size_mb:.2f} МБ)")



# Сохранение весов LoRA
save_lora_weights(model, 'techstartup_lora.pt')

# Сравнение с полным размером модели
torch.save(model.state_dict(), 'full_model.pt')
full_size = os.path.getsize('full_model.pt') / (1024 * 1024)
lora_size = os.path.getsize('techstartup_lora.pt') / (1024 * 1024)
print(f"Полная модель сохранена: full_model.pt ({full_size:.1f} МБ)")
print(f"\nLoRA в {full_size/lora_size:.0f}x меньше, чем полная модель!")

In [ ]:
def load_lora_weights(model, filepath):
    """
    Загрузка весов LoRA в модель, у которой уже есть слои LoRA.
    """
    lora_state_dict = torch.load(filepath, weights_only=True)
    model.load_state_dict(lora_state_dict, strict=False)
    print(f"Веса LoRA загружены из: {filepath}")



# Демонстрация загрузки
print("\n===== Тест сохранения/загрузки =====")
# Адаптеру нужны те же базовые веса: сначала восстановите веса из best_model.pt,
# затем создайте те же слои LoRA и загрузите веса адаптера.
new_model = MiniGPT(config).to(device)
new_model.load_state_dict(checkpoint["model_state_dict"])  # сначала база!
new_model = add_lora_to_model(new_model, rank=8, alpha=16)

before_load = generate_response(new_model, "Чем занимается TechStartup Inc?", tokenizer)
print(f"До загрузки LoRA: {before_load[:60]}...")

load_lora_weights(new_model, 'techstartup_lora.pt')

after_load = generate_response(new_model, "Чем занимается TechStartup Inc?", tokenizer)
print(f"После загрузки LoRA:  {after_load[:60]}...")
# Проверка: перезагруженная модель должна давать те же логиты, что обученная
probe = tokenizer.encode("[INST] Что такое SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval(); new_model.eval()
with torch.no_grad():
    torch.testing.assert_close(new_model(probe), model(probe), atol=1e-5, rtol=0)
print("Проверка совпадения после перезагрузки пройдена")
print("\nАдаптеры LoRA успешно сохранены и загружены!")


# ===== СЛИЯНИЕ: вшиваем LoRA в базу и УДАЛЯЕМ обёртки =====
def merge_lora_into_base(model):
    """
    Вшить LoRA в базовые веса и УДАЛИТЬ обёртки адаптеров,
    чтобы слитая модель снова стала обычным MiniGPT (необратимо).
    """
    for block in model.blocks:
        lora = block.attn.qkv_proj
        if isinstance(lora, LoRALinear):
            # Вшиваем поправку в базовый вес: W + (A @ B * scale).T
            delta = lora.lora_A @ lora.lora_B * lora.scale
            lora.base.weight.data += delta.T  # Транспонирование под соглашение Linear
            # Меняем обёртку на обычный слитый слой
            block.attn.qkv_proj = lora.base

    print("Веса LoRA слиты с базовой моделью")

# Сливаем и проверяем: те же логиты, параметров адаптеров больше нет.
# ВНИМАНИЕ: слияние необратимо; запускайте эту ячейку только после сравнений выше.
probe = tokenizer.encode("[INST] Что такое SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    adapter_logits = model(probe)

merge_lora_into_base(model)

with torch.no_grad():
    merged_logits = model(probe)
torch.testing.assert_close(merged_logits, adapter_logits, atol=1e-5, rtol=0)

total = sum(p.numel() for p in model.parameters())
print(f"Параметры слитой модели: {total:,} (адаптеры удалены)")
assert not any(isinstance(m, LoRALinear) for m in model.modules())
print("Проверка совпадения после слияния пройдена")
# Почему транспонирование? forward вычисляет x @ A @ B, поправка имеет форму (in, out);
# nn.Linear хранит веса как (out, in), отсюда delta.T. Замена обёртки (а не просто
# обнуление адаптеров) убирает два мёртвых матричных умножения адаптеров.


## Резюме

**Что мы создали:**

1. **Набор данных инструкция/ответ** с маскированием потерь
2. **Адаптеры LoRA**, которые обучают <1% параметров
3. **Цикл дообучения** с использованием 5-шагового рецепта
4. **Оценку**, сравнивающую поведение до/после (честно: потери снижаются, текст остаётся бессмыслицей на базе, обученной с нуля)
5. **Сохранение/загрузку адаптеров** для развёртывания

**Ключевые инсайты:**

- Маскирование потерь фокусирует обучение на том, что важно (ответы, а не инструкции)
- LoRA достигает отличных результатов с 48× меньшим количеством обучаемых параметров
- Замораживание базовых весов значительно снижает **катастрофическое забывание** (потерю предобученных знаний)
- Качество данных важнее количества для дообучения
- Адаптеры крошечные и могут быть заменены для разных задач

**Далее:** Глава 14 исследует промпт-инженерию, получение лучших результатов без изменения весов модели вообще!

## Необязательный эксперимент: вариант со случайной базой (явно помечен)

Основной поток выше везде использует чекпоинт главы 12. Этот эксперимент повторяет рецепт LoRA на **случайно инициализированной базе**, чтобы показать, почему чекпоинт важен. Это НЕ прогон книги: база, обучение и перезагрузка здесь делят одни случайные веса, отдельно от потока с чекпоинтом.

In [ ]:
# ===== ВАРИАНТ СО СЛУЧАЙНОЙ БАЗОЙ (помеченный эксперимент, НЕ прогон книги) =====
set_seed(42)
rand_base = MiniGPT(config).to(device)  # Случайные веса, без чекпоинта
rand_model = MiniGPT(config).to(device)
rand_model.load_state_dict(rand_base.state_dict())  # Обе модели начинают с одних и тех же случайных весов
rand_model = add_lora_to_model(rand_model, rank=8, alpha=16)

rand_optimizer = AdamW([p for p in rand_model.parameters() if p.requires_grad],
                       lr=1e-3, weight_decay=0.01)
rand_losses = []
for epoch in range(3):
    loss = finetune_epoch(rand_model, train_loader, rand_optimizer, device)
    rand_losses.append(loss)
    print(f"[Случайная база] Эпоха {epoch+1}: потери по немаскированным токенам ответа {loss:.4f}")

print(f"\nПотери при старте с чекпоинта: {[f'{l:.4f}' for l in train_losses]}")
print(f"Потери при случайной инициализации: {[f'{l:.4f}' for l in rand_losses]}")
print("Порядок обучения тот же, но потери зависят от исходных весов.")
print("Поэтому загружайте адаптер на ту базовую модель, на которой его обучили.")

## Упражнения

### Упражнение 1: Разные ранги LoRA

Попробуйте rank=4 и rank=16. Как это влияет на обучение и результаты?

In [ ]:
# ВАШ КОД ЗДЕСЬ
# 1. Создайте модель с rank=4
# 2. Обучите в течение 3 эпох
# 3. Сравните потери и оценку с rank=8

### Упражнение 2: Создайте свой собственный FAQ

Создайте набор данных о теме, которую вы знаете (ваша школа, хобби и т.д.).

In [ ]:
# ВАШ КОД ЗДЕСЬ
# 1. Создайте 30+ пар инструкция/ответ
# 2. Выполните дообучение модели
# 3. Протестируйте со своими вопросами

### Упражнение 3: Аблация маскирования потерь

Что произойдёт, если мы НЕ будем маскировать токены инструкции?

In [ ]:
# ВАШ КОД ЗДЕСЬ
# 1. Измените набор данных, чтобы НЕ маскировать токены инструкции (все метки = фактические токены)
# 2. Обучите модель
# 3. Сравните результаты с маскированной версией
# 4. Что вы наблюдаете?